# **TITLE, PURPOSE, CONFIGURATIONS**

In [1]:
# Notebook 2 — Cell 1: Title, imports, and config
import os
import pandas as pd
import numpy as np
import random
import warnings

# silence noisy warnings for a cleaner notebook
warnings.filterwarnings("ignore")

# ----------------------
# Configuration constants
# ----------------------
RANDOM_STATE = 123                # reproducible seed for randomness
VAL_DAYS = 60                     # number of most-recent days to hold out for validation
STAGE5_POLICY = "keep"            # choices: "keep", "downweight", "downsample"
MODELING_TABLE_PATH = "data/modeling_table.csv"
MODEL_SAVE_PATH = "models/retail_demand_xgb_best.pkl"
RESULTS_DIR = "results"

# create output directories if they don't exist
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

# echo the configuration so you can confirm before proceeding
print("Notebook 2 — Modeling config")
print(f"  RANDOM_STATE  = {RANDOM_STATE}")
print(f"  VAL_DAYS      = {VAL_DAYS}")
print(f"  STAGE5_POLICY = {STAGE5_POLICY}")
print(f"  MODELING_TABLE_PATH = {MODELING_TABLE_PATH}")
print(f"  MODEL_SAVE_PATH     = {MODEL_SAVE_PATH}")
print(f"  RESULTS_DIR         = {RESULTS_DIR}")

Notebook 2 — Modeling config
  RANDOM_STATE  = 123
  VAL_DAYS      = 60
  STAGE5_POLICY = keep
  MODELING_TABLE_PATH = data/modeling_table.csv
  MODEL_SAVE_PATH     = models/retail_demand_xgb_best.pkl
  RESULTS_DIR         = results


# LOADING DATA and APPLY STAGE-5 Policy

In [2]:
import pandas as pd

df = pd.read_csv(MODELING_TABLE_PATH, parse_dates = ['date'])

print("Loaded modeling table:", MODELING_TABLE_PATH)
print("Shape:", df.shape)
display("\nColumns:", df.columns.tolist())

#now we ensure date is datetime
df['date'] = pd.to_datetime(df['date'])


Loaded modeling table: data/modeling_table.csv
Shape: (262080, 24)


'\nColumns:'

['sku_id',
 'date',
 'units_sold',
 'lag_7_units_sold',
 'lag_14_units_sold',
 'rolling_7d_avg',
 'days_since_launch',
 'inventory_ratio',
 'discount_pct',
 'days_to_season_end',
 'season',
 'is_holiday',
 'markdown_stage',
 'category_jackets',
 'category_jeans',
 'category_shoes',
 'category_tshirts',
 'dow_Friday',
 'dow_Monday',
 'dow_Saturday',
 'dow_Sunday',
 'dow_Thursday',
 'dow_Tuesday',
 'dow_Wednesday']

**Now let us check for common leakage columns that should npt be in training data"**

**then if we find any exist, we drop them**



In [3]:
leakage_cols = ['latent_demand', 'markdown_depth', 'days_at_current_price']
present_leakage = [c for c in leakage_cols if c in df.columns]
if present_leakage:
    print("\nwarning: Found leakage columns - dropping from df:", present_leakage)
    df = df.drop(columns = present_leakage)

#let us do any null check 
null_summary = df.isna().sum()
null_cols = null_summary[null_summary > 0]. sort_values(ascending = False)
if len(null_cols):
    print("\nColumns with nulls (count):")
    print(null_cols)
else:
    print("\nNo nulls detected in the table.")



Columns with nulls (count):
lag_14_units_sold    7000
lag_7_units_sold     3500
rolling_7d_avg        500
dtype: int64


**Stage /segment distributions: useful to confimr ealrier findings**


In [4]:
print("\nmarkdown_stage value counts:")
print(df['markdown_stage'].value_counts().sort_index())

if 'popularity_segment' in df.columns:
    print("\npopularity_segment value counts:")
    print(df['popularity_segment'].value_counts())

# Cross-tab quick view (markdown_stage × popularity_segment)
if 'popularity_segment' in df.columns:
    print("\nRow counts by markdown_stage × popularity_segment (first 20 rows):")
    print(pd.crosstab(df['markdown_stage'], df['popularity_segment']).head(20))

# Add policy helper columns for training-time handling of stage 5
# Default sample_weight = 1.0 (no weighting)
df['sample_weight'] = 1.0


markdown_stage value counts:
markdown_stage
0     22285
1     22476
2     23728
3     22544
4     22517
5    148530
Name: count, dtype: int64


### Stage‑5 Handling: Downweighting vs Downsampling

In our dataset, **markdown_stage = 5** (deep clearance) is heavily overrepresented.  
This is realistic in retail, but it can cause the model to become **biased toward late‑stage behavior**, hurting accuracy in earlier stages where pricing decisions matter most.

To address this, we support three policies:

---

## 1. **Keep‑as‑is** (default)
We keep all rows with equal weight.  
This preserves the natural distribution and is a good starting point for baseline models.

---

## 2. **Downweighting** (keep all rows, but reduce influence)
Downweighting means:

- Stage‑5 rows stay in the dataset  
- But they count *less* during training  
- Example: weight = 0.2 → each stage‑5 row counts as 20% of a normal row

This helps when we want to keep all data but prevent stage‑5 from dominating the loss function.  
Models like **XGBoost** and **Linear Regression** can use `sample_weight` directly.

**Code behavior:**



In [5]:
# If STAGE5_POLICY == 'downweight' -> assign smaller weight to stage 5 rows
if STAGE5_POLICY == "downweight":
    # choose a downweight factor (you can change later). Here we use 0.2 (80% downweight).
    DOWNWEIGHT_FACTOR = 0.2
    df.loc[df['markdown_stage'] == 5, 'sample_weight'] = DOWNWEIGHT_FACTOR
    print(f"\nApplied downweighting to stage 5 rows: factor={DOWNWEIGHT_FACTOR}")

# For downsample, we add a 'downsample_prob' column that will be used AFTER the time split
# to randomly drop stage-5 rows from the training pool (validation remains untouched).
if STAGE5_POLICY == "downsample":
    # fraction of stage-5 rows to keep in training (tunable)
    DOWNSAMPLE_FRAC = 0.2
    # Default keep prob = 1 for non-stage5
    df['downsample_prob'] = 1.0
    df.loc[df['markdown_stage'] == 5, 'downsample_prob'] = DOWNSAMPLE_FRAC
    print(f"\nMarked stage 5 rows with downsample probability = {DOWNSAMPLE_FRAC}")
else:
    # create column with keep prob = 1.0 so later code can always rely on its existence
    df['downsample_prob'] = 1.0

# Final quick head and a confirmation print
print("\nPreview (first 3 rows):")
print(df.head(3))

print("\nCell 2 complete: data loaded, leakages removed (if any), and policy markers added.")


Preview (first 3 rows):
     sku_id       date  units_sold  lag_7_units_sold  lag_14_units_sold  \
0  SKU_0001 2022-01-01           8               NaN                NaN   
1  SKU_0001 2022-01-02           4               NaN                NaN   
2  SKU_0001 2022-01-03           5               NaN                NaN   

   rolling_7d_avg  days_since_launch  inventory_ratio  discount_pct  \
0             NaN                  0         0.982222           0.0   
1             8.0                  1         0.973333           0.0   
2             6.0                  2         0.962222           0.0   

   days_to_season_end  ... category_tshirts  dow_Friday  dow_Monday  \
0                  58  ...                0           0           0   
1                  57  ...                0           0           0   
2                  56  ...                0           0           1   

   dow_Saturday  dow_Sunday  dow_Thursday  dow_Tuesday  dow_Wednesday  \
0             1           0    

### Cell: Time-based train/validation split and apply stage-5 downsampling (if chosen)

Goal: Hold out the last `VAL_DAYS` as validation (simulate future data). If STAGE5_POLICY == "downsample", randomly drop a fraction of stage-5 rows from the training pool using a reproducible RNG. We do NOT touch validation rows so evaluation reflects the true future distribution.

What to check after running:
- Train / Val row counts and date ranges
- Distribution of `markdown_stage` in train vs val
- If downsampling was applied: number of stage-5 rows kept vs dropped
- That `sample_weight_train` is available for model.fit

In [6]:
#Cell 3A: compute cutoff and create train / val splits
cutoff = df['date'].max() - pd.Timedelta(days=VAL_DAYS)
train_df = df[df['date'] <= cutoff].copy().reset_index(drop=True)
val_df   = df[df['date'] >  cutoff].copy().reset_index(drop=True)

print("Train rows:", train_df.shape[0])
print("Val   rows:", val_df.shape[0])
print("Train max date:", train_df['date'].max())
print("Val   min date:", val_df['date'].min())

Train rows: 241200
Val   rows: 20880
Train max date: 2023-11-01 00:00:00
Val   min date: 2023-11-02 00:00:00


**Quick distributions & sanity checks**

In [7]:
# Cell 3B: markdown_stage and popularity_segment distributions (train vs val)
print("markdown_stage distribution — train:")
print(train_df['markdown_stage'].value_counts().sort_index())

print("\nmarkdown_stage distribution — val:")
print(val_df['markdown_stage'].value_counts().sort_index())

if 'popularity_segment' in train_df.columns:
    print("\npopularity_segment distribution — train:")
    print(train_df['popularity_segment'].value_counts())
    print("\nCross-tab (markdown_stage × popularity_segment) — train (small view):")
    print(pd.crosstab(train_df['markdown_stage'], train_df['popularity_segment']).head())

markdown_stage distribution — train:
markdown_stage
0     22285
1     22476
2     23728
3     22544
4     22517
5    127650
Name: count, dtype: int64

markdown_stage distribution — val:
markdown_stage
5    20880
Name: count, dtype: int64


**Apply reproducible downsampling to training (only if policy = downsample)**

In [8]:
# Cell 3C: apply downsampling to training set if STAGE5_POLICY == "downsample"
import numpy as np

if STAGE5_POLICY == "downsample":
    rng = np.random.RandomState(RANDOM_STATE)
    before_total = len(train_df)
    before_stage5 = train_df['markdown_stage'].eq(5).sum()
    keep_mask = rng.rand(len(train_df)) < train_df['downsample_prob'].values
    train_df = train_df.loc[keep_mask].reset_index(drop=True)
    after_total = len(train_df)
    after_stage5 = train_df['markdown_stage'].eq(5).sum()
    print(f"Downsample applied: total {before_total} -> {after_total}; stage5 {before_stage5} -> {after_stage5}")
else:
    print("STAGE5_POLICY != 'downsample' — no sampling applied to training set.")

STAGE5_POLICY != 'downsample' — no sampling applied to training set.


### Preparing X_train, X_val, y_train, y_val, and sample weights

In this step we build the actual inputs our models will use.

- **FEATURES**: all predictive columns (we exclude identifiers, the target, and helper columns like `downsample_prob`).
- **X_train / X_val**: feature matrices for training and validation.
- **y_train / y_val**: target vectors as NumPy arrays.
- **sample_weight_train**: row weights used only if we applied the *downweight* policy (otherwise all weights = 1).

**Why `.copy()`?**  
To create clean, independent DataFrames and avoid pandas slice warnings.

**Why `.values`?**  
To convert the target into a NumPy array, which ML models expect.

This prepares the data in the correct format for Linear Regression, Random Forest, and XGBoost.


In [9]:
# Cell 3D — prepare FEATURES, handle NaNs, one-hot encode season, build X/y

TARGET = "units_sold"
excluded_cols = {'sku_id', 'date', TARGET, 'downsample_prob'}

# 1. Drop rows with missing lag/rolling values
train_df = train_df.dropna().reset_index(drop=True)
val_df   = val_df.dropna().reset_index(drop=True)

# 2. Combine train + val BEFORE one-hot encoding
combined = pd.concat([train_df, val_df], axis=0)

# 3. One-hot encode season ONCE
combined = pd.get_dummies(combined, columns=['season'], drop_first=False)

# 4. Split back into train_df and val_df
train_df = combined.iloc[:len(train_df)].reset_index(drop=True)
val_df   = combined.iloc[len(train_df):].reset_index(drop=True)

# 5. Build FEATURES list
FEATURES = [c for c in train_df.columns 
            if c not in excluded_cols and c != 'sample_weight']

# 6. Build X/y
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].values

X_val   = val_df[FEATURES].copy()
y_val   = val_df[TARGET].values

sample_weight_train = train_df['sample_weight'].values

print("Features count:", len(FEATURES))
print("Example features:", FEATURES[:10])
print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape, "y_val shape:", y_val.shape)
print("sample_weight_train shape:", sample_weight_train.shape)


Features count: 24
Example features: ['lag_7_units_sold', 'lag_14_units_sold', 'rolling_7d_avg', 'days_since_launch', 'inventory_ratio', 'discount_pct', 'days_to_season_end', 'is_holiday', 'markdown_stage', 'category_jackets']
X_train shape: (234200, 24) y_train shape: (234200,)
X_val shape: (20880, 24) y_val shape: (20880,)
sample_weight_train shape: (234200,)


### Baseline 1 — Linear Regression

What this cell does:
- Fit a plain Linear Regression on X_train / y_train (honoring `sample_weight_train`).
- Predict on X_val and compute RMSE, MAE, RMSLE.
- Print overall metrics and a brief per-segment breakdown (avg actual vs avg predicted by popularity_segment).
- Save the results to `results/baseline_metrics.csv` (append mode) so we can compare models later.

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert 'X_train' in globals() and 'X_val' in globals(), "Run previous cells (3) to prepare X/y first."
assert len(X_train) > 0 and len(X_val) > 0, "Empty train/val — check previous split."


In [11]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rmsle(y_true, y_pred):
    y_pred_nonneg = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred_nonneg)))

# Drop rows with missing lag/rolling values
df = df.dropna().reset_index(drop=True)
print("After dropping NaNs:", df.shape)



After dropping NaNs: (255080, 26)


# Now we fit the Linear Regression Model

In [12]:
lr = LinearRegression()
lr.fit(X_train, y_train, sample_weight=sample_weight_train)




,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rmsle(y_true, y_pred):
    y_pred_nonneg = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred_nonneg)))

# predict on validation
y_pred_lr = lr.predict(X_val)

# clip negatives for metrics
y_pred_lr_clipped = np.clip(y_pred_lr, 0, None)

print("Linear Regression — validation metrics (clipped preds):")
print("  RMSE :", round(rmse(y_val, y_pred_lr_clipped), 4))
print("  MAE  :", round(mean_absolute_error(y_val, y_pred_lr_clipped), 4))
print("  RMSLE:", round(rmsle(y_val, y_pred_lr_clipped), 4))
print("  mean_true:", round(np.mean(y_val), 4),
      "mean_pred:", round(np.mean(y_pred_lr_clipped), 4))


Linear Regression — validation metrics (clipped preds):
  RMSE : 1.3611
  MAE  : 0.5956
  RMSLE: 0.5956
  mean_true: 0.0003 mean_pred: 0.5957
